# GOAD 논문 기반 변환 앙상블

https://arxiv.org/html/2005.02359v1

논문 식 3 중심 triplet + 변환 분류 교차엔트로피, 식 4 epsilon 확률, 식 5 음의 로그확률 합을 구현. Gaussian 선형 변환 16개, 투영 8차원, 작은 8→4 MLP; 1회/8회 학습 후보. 논문 전체 설정 재현이 아닌 소규모 센서 적용. 두 시드 평균, 두 관점 최대 앙상블. 정상만 표현 학습, 임계값/모델 선택은 내부 구간 검증. 논문의 테스트 불량 수로 임계값을 맞추는 평가 방식은 사용하지 않음.

동일 RH 정상/가스 1325개, 10개 관측구간 중첩 검증. 기존에 본 자료라 새 독립 검증 아님. 실제 Run 확정 구분 아님. 새로운 데이터에서 성공할 때까지의 성능 보장 없음.


In [1]:
from pathlib import Path
import json,copy,numpy as np,pandas as pd,torch
from torch import nn
from sklearn.metrics import confusion_matrix
torch.set_num_threads(1)
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/74_product_normalized_joint_gas_modes.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.rsplit('memo={}',1)[0])
memo={};logs=[]
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 per_view=[]
 for v,(tr,te) in enumerate(views(ref,q)):
  tr=torch.tensor(np.clip(tr,-8,8),dtype=torch.float32)
  te=torch.tensor(np.clip(te,-8,8),dtype=torch.float32)
  normal=tr[y[ref]==0]
  seed_scores=[]
  for seed in [42,7]:
   torch.manual_seed(seed)
   tasks=16;r=8
   W=torch.randn(tasks,tr.shape[1],r)
   net=nn.Sequential(nn.Linear(r,8),nn.LeakyReLU(.1),nn.Linear(8,4))
   head=nn.Linear(4,tasks)
   opt=torch.optim.Adam(list(net.parameters())+list(head.parameters()),lr=.001)
   scores=[]
   for epoch in range(1,9):
    net.train()
    for idx in torch.randperm(len(normal)).split(128):
     z=net(torch.einsum('nd,tdr->ntr',normal[idx],W))
     center=z.mean(0)
     dist=((z[:,:,None,:]-center[None,None,:,:])**2).sum(-1)
     own=dist.diagonal(dim1=1,dim2=2)
     other=dist.masked_fill(torch.eye(tasks,dtype=torch.bool)[None],float('inf')).min(-1).values
     target=torch.arange(tasks).repeat(len(idx))
     loss=(own+.1-other).clamp_min(0).mean()+nn.functional.cross_entropy(head(z).reshape(-1,tasks),target)+1e-4*z.square().mean()
     opt.zero_grad();loss.backward();opt.step()
    if epoch in [1,8]:
     net.eval()
     with torch.no_grad():
      centers=net(torch.einsum('nd,tdr->ntr',normal,W)).mean(0)
      def score(x):
       z=net(torch.einsum('nd,tdr->ntr',x,W))
       dist=((z[:,:,None,:]-centers[None,None,:,:])**2).sum(-1)
       # Exact epsilon-regularized equation 4 in log-space, not shifted-distance softmax.
       logmass=torch.logaddexp(-dist,torch.tensor(np.log(1e-8),dtype=torch.float32))
       logp=logmass-torch.logsumexp(logmass,dim=-1,keepdim=True)
       return -logp.diagonal(dim1=1,dim2=2).sum(1).numpy()
      cal=score(normal);out=score(te)/max(float(np.quantile(cal,.95)),1e-6)
     assert np.isfinite(out).all()
     scores.append(out)
     logs.append(dict(train=len(ref),query=len(q),view=v,seed=seed,epoch=epoch,tasks=tasks,normal_training=len(normal),gas_training=0,loss=float(loss.detach()),train_blocks=','.join(map(str,groups.iloc[ref].unique())),query_blocks=','.join(map(str,groups.iloc[q].unique()))))
   seed_scores.append(np.column_stack(scores))
  per_view.append(np.mean(seed_scores,axis=0))
 memo[key]=np.max(per_view,axis=0)
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 outer=predict(ref,q);inner=np.full((len(ref),2),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['GOAD_epoch1','GOAD_epoch8']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   ip=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],ip,labels=[0,1]).ravel()
   recall=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((recall,f1,-fp,name,budget,outer[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(outer[j,k]>cut),score=float(outer[j,k]),threshold=cut))
 selected=max(candidates,key=lambda a:a[:3]);choices.append(dict(block=int(b),selected=selected[3],budget=selected[4],inner_recall=selected[0]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(selected[5][j]),score=np.nan,threshold=np.nan))
 print('Finished GOAD block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget,product),g in p.groupby(['model','budget','product']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,product=product,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn) if tp+fn else np.nan,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(logs),'training_audit')]:
 obj.to_csv(tables/f'gas_goad_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))



CN7 extra metadata 713 sensor-consistent unique inputs 713 conflicting 0


Finished GOAD block 1


Finished GOAD block 2


Finished GOAD block 3


Finished GOAD block 4


...

Finished GOAD block 7


Finished GOAD block 8


Finished GOAD block 9


Finished GOAD block 10


                      model  budget product  TP  FP  FN  TN   recall       F1
                GOAD_epoch1    0.02     CN7   0  14  10 681 0.000000 0.000000
                GOAD_epoch1    0.02     RG3   1  10  18 591 0.052632 0.066667
                GOAD_epoch1    0.05     CN7   0  60  10 635 0.000000 0.000000
                GOAD_epoch1    0.05     RG3   1  17  18 584 0.052632 0.054054
                GOAD_epoch1    0.10     CN7   0 102  10 593 0.000000 0.000000
                GOAD_epoch1    0.10     RG3   1  49  18 552 0.052632 0.028986
                GOAD_epoch8    0.02     CN7   0  20  10 675 0.000000 0.000000
                GOAD_epoch8    0.02     RG3   1  10  18 591 0.052632 0.066667
                GOAD_epoch8    0.05     CN7   0  63  10 632 0.000000 0.000000
                GOAD_epoch8    0.05     RG3   1  19  18 582 0.052632 0.051282
                GOAD_epoch8    0.10     CN7   0 114  10 581 0.000000 0.000000
                GOAD_epoch8    0.10     RG3   1  50  18 551 0.05